<a href="https://colab.research.google.com/github/eflsullivan/seqADAGE/blob/master/2026_08_11_gene_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gene Filtering Notebook

* filters out (by strain) any genes not in gene_list.csv

# 0. Connect to GitHub

In [ ]:
!git clone https://github.com/sullive2/seqADAGE.git

fatal: destination path 'seqADAGE' already exists and is not an empty directory.


In [ ]:
%cd seqADAGE/Py/phylogenetic_learning

/content/seqADAGE/Py/phylogenetic_learning


In [ ]:
!pip install -qq tensorflow keras statsmodels seaborn plotnine_prism pydeseq2

# 1. Loading classes and modules

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
# ADAGE
from adage import Adage as ad
from adage import SeqAdage

# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
from plotnine_prism import *

# Miscellaneous
import time
import tensorflow as tf
from pydeseq2.preprocessing import deseq2_norm

In [ ]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  0
True


# 2. Loading Files

In [ ]:
# load Copy of num_reads_ecoli_pan_genome_part2_comp_ref_k15v3.csv

pan_genome = '/content/drive/My Drive/Comp-Bio-Projs/data/Emma/Copy of num_reads_ecoli_pan_genome_part2_comp_ref_k15v3.csv'
pan_genome_df = pd.read_csv(pan_genome)

In [ ]:
pan_genome_df.head()

,Unnamed: 0,X,Name,ERX358242.salmon,ERX358243.salmon,ERX358244.salmon,ERX358245.salmon,ERX358246.salmon,ERX358247.salmon,ERX358248.salmon,...,SRX28568280.salmon,SRX28568281.salmon,SRX28568282.salmon,SRX28568283.salmon,SRX28568284.salmon,SRX28568285.salmon,SRX28568286.salmon,SRX28568287.salmon,SRX28568288.salmon,SRX28568289.salmon
0,1,0ed86576d343ce011a61773e0620e335_1041,NaN,0.0,22.519,0.0,0.000,0.000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000,32.0,0.000,30.000,0.0,0.0
1,2,0ed86576d343ce011a61773e0620e335_1045,NaN,15.0,10.000,56.0,8.000,25.000,2.0,6.0,...,0.0,0.0,0.0,3.0,2.000,0.0,0.000,0.000,0.0,0.0
2,3,0ed86576d343ce011a61773e0620e335_1047,NaN,0.0,0.000,0.0,8.188,8.165,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.000,0.0,0.0
3,4,0ed86576d343ce011a61773e0620e335_1120,NaN,21.0,26.000,20.0,4.000,10.000,0.0,7.0,...,61.0,25.0,15.0,29.0,17.000,33.0,19.000,15.000,22.0,27.0
4,5,0ed86576d343ce011a61773e0620e335_1122,NaN,15.0,25.000,36.0,7.000,21.000,7.0,28.0,...,36.0,14.0,3.0,12.0,22.851,41.0,21.917,15.913,7.0,30.0


In [ ]:
gpa = '/content/drive/My Drive/Comp-Bio-Projs/data/Emma/ecoli_pan_genomev3_gene_presence_absence_ann.csv'
gpa_df = pd.read_csv(gpa)

/tmp/ipykernel_13492/1953879303.py:2: DtypeWarning: Columns (0: W3110, 1: UPEC_536, 2: BW25113_K12, 3: BW25113_AM187, 4: BL21, 5: BL21_DE3) have mixed types. Specify dtype option on import or set low_memory=False.


In [ ]:
gpa_df.head(2)

,Gene,Non.unique.Gene.name,Annotation,No..isolates,No..sequences,Avg.sequences.per.isolate,Genome.Fragment,Order.within.Fragment,Accessory.Fragment,Accessory.Order.with.Fragment,...,BW25113_AM187,BL21,BL21_DE3,panDB,compound_name,first_name,compound_name_comp,first_name_comp,hyp_comp,karp
0,yaaA,NaN,peroxide stress protein YaaA,19,19,1.0,23654,14647,NaN,NaN,...,E1A98_RS00030,HO397_RS00030,HO396_RS00030,panDB562_10205,b0006 ECs_0006 Y75_RS00035 UTI89_RS00035 ECP_R...,b0006,11997cc26382b2c286cd502685a104a5_13 e0ba967783...,11997cc26382b2c286cd502685a104a5_13,named,WELL-CHARACTERIZED
1,nhaR,NaN,transcriptional activator NhaR,19,19,1.0,23654,14623,NaN,NaN,...,E1A98_RS00090,HO397_RS00090,HO396_RS00090,panDB562_10193,b0020 ECs_0018 Y75_RS00095 UTI89_RS00110 ECP_R...,b0020,11997cc26382b2c286cd502685a104a5_45 e0ba967783...,11997cc26382b2c286cd502685a104a5_45,named,WELL-CHARACTERIZED


In [ ]:
path = '/content/drive/My Drive/Comp-Bio-Projs/data/Emma/'
gene_list = 'gene_list.csv'
gene_list_df = pd.read_csv(path + gene_list)

gene_list_df.head()

,Unnamed: 0,MG1655,EPEC,EHEC
0,0,kefC,WP_000979516.1,acrA
1,1,acrA,WP_001115524.1,mscK
2,2,acrR,group_21221,dnaX
3,3,mscK,yeeO,ybaB
4,4,rsmS,WP_000490679.1,folA


# 3. Gene Filtering

In [ ]:
# makes list of gene ids from compound_name_comp col in gpa file
# adds ids to list

def comp_name_list(strain):
  comp_name_list_full = []
  for gene in gpa_df['Gene']:
    if gene in gene_list_df[strain].values:
      row_idx = gpa_df.index.get_loc(gpa_df[gpa_df['Gene'] == gene].index[0])
      value = gpa_df.at[row_idx, 'compound_name_comp']
      comp_name_list_full.append(value)
  comp_name_list_full = [words for segments in comp_name_list_full for words in segments.split()]
  return comp_name_list_full

comp_name_list_full = comp_name_list('MG1655')

In [ ]:
comp_name_list_full = comp_name_list('EPEC')

In [ ]:
comp_name_list_full = comp_name_list('EHEC')

In [ ]:
len(comp_name_list_full)

64199

In [ ]:
len(pan_genome_df)

59521

In [ ]:
# makes list of row indices containing genes in pan_genome csv and in comp_name_list_full

row_idx_list = []

for gene_id in comp_name_list_full:
  if gene_id in pan_genome_df['X'].values:
    row_idx = pan_genome_df.index.get_loc(pan_genome_df[pan_genome_df['X'] == gene_id].index[0])
    row_idx_list.append(row_idx)

In [ ]:
# copies rows from pan_genome into new dataframe if their index is in comp_name_list_full

final_df = pan_genome_df.loc[row_idx_list].copy()

In [ ]:
len(pan_genome_df)

59521

In [ ]:
len(final_df)

5030

In [ ]:
final_df.head()

,Unnamed: 0,X,Name,ERX358242.salmon,ERX358243.salmon,ERX358244.salmon,ERX358245.salmon,ERX358246.salmon,ERX358247.salmon,ERX358248.salmon,...,SRX28568280.salmon,SRX28568281.salmon,SRX28568282.salmon,SRX28568283.salmon,SRX28568284.salmon,SRX28568285.salmon,SRX28568286.salmon,SRX28568287.salmon,SRX28568288.salmon,SRX28568289.salmon
241,242,11997cc26382b2c286cd502685a104a5_13,NaN,89.000,210.000,184.000,97.000,138.000,147.000,372.990,...,2456.00,1020.000,2966.000,2871.000,832.000,1343.000,1036.000,948.000,1176.000,1315.000
1853,1854,11997cc26382b2c286cd502685a104a5_45,NaN,190.000,247.000,353.000,111.000,263.000,108.000,326.000,...,2505.00,1812.000,506.250,647.000,501.000,761.877,563.000,611.659,599.889,755.000
104,105,11997cc26382b2c286cd502685a104a5_102,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,1653.22,959.272,1127.387,1297.846,1298.371,1737.095,1281.192,1130.493,1107.438,991.557
145,146,11997cc26382b2c286cd502685a104a5_110,NaN,512.188,774.683,494.736,375.694,411.855,148.855,1310.018,...,5772.00,3436.000,2976.000,4803.000,2792.000,2721.000,2274.000,2376.000,2393.000,2761.000
3205,3206,11997cc26382b2c286cd502685a104a5_72,NaN,364.959,1070.884,348.464,655.238,418.039,217.520,1457.939,...,105679.00,54749.000,1928.000,1996.000,8069.000,10965.000,8550.860,8671.000,105943.000,72627.000


In [ ]:
final_df.drop(columns=['Unnamed: 0'])

,X,Name,ERX358242.salmon,ERX358243.salmon,ERX358244.salmon,ERX358245.salmon,ERX358246.salmon,ERX358247.salmon,ERX358248.salmon,ERX358249.salmon,...,SRX28568280.salmon,SRX28568281.salmon,SRX28568282.salmon,SRX28568283.salmon,SRX28568284.salmon,SRX28568285.salmon,SRX28568286.salmon,SRX28568287.salmon,SRX28568288.salmon,SRX28568289.salmon
241,11997cc26382b2c286cd502685a104a5_13,NaN,89.000,210.000,184.000,97.000,138.000,147.000,372.990,115.000,...,2456.00,1020.000,2966.000,2871.000,832.000,1343.000,1036.000,948.000,1176.000,1315.000
1853,11997cc26382b2c286cd502685a104a5_45,NaN,190.000,247.000,353.000,111.000,263.000,108.000,326.000,104.000,...,2505.00,1812.000,506.250,647.000,501.000,761.877,563.000,611.659,599.889,755.000
104,11997cc26382b2c286cd502685a104a5_102,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,1653.22,959.272,1127.387,1297.846,1298.371,1737.095,1281.192,1130.493,1107.438,991.557
145,11997cc26382b2c286cd502685a104a5_110,NaN,512.188,774.683,494.736,375.694,411.855,148.855,1310.018,278.331,...,5772.00,3436.000,2976.000,4803.000,2792.000,2721.000,2274.000,2376.000,2393.000,2761.000
3205,11997cc26382b2c286cd502685a104a5_72,NaN,364.959,1070.884,348.464,655.238,418.039,217.520,1457.939,169.932,...,105679.00,54749.000,1928.000,1996.000,8069.000,10965.000,8550.860,8671.000,105943.000,72627.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8008,e0ba967783ad45e29471d6259beaaa64_10659,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8037,e0ba967783ad45e29471d6259beaaa64_10758,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8491,e0ba967783ad45e29471d6259beaaa64_2747,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8683,e0ba967783ad45e29471d6259beaaa64_3506,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


# 4. Exports dataframes to Google Drive

* (Comp-Bio-Projs > data > Emma)

In [ ]:
final_df.to_csv('/content/drive/My Drive/Comp-Bio-Projs/data/Emma/EHEC_filtered_genes.csv', index=False)
final_df.to_csv('/content/drive/My Drive/Comp-Bio-Projs/data/Emma/EPEC_filtered_genes.csv', index=False)
final_df.to_csv('/content/drive/My Drive/Comp-Bio-Projs/data/Emma/MG1655_filtered_genes.csv', index=False)